In [1]:
!pip install tqdm


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from tqdm import tqdm
print("tqdm installed successfully")

tqdm installed successfully


In [3]:
!pip install ImageHash


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path

# Core Paths
RAW_IMAGE_DIR  = Path("../data/raw/skin_images")
PROCESSED_DIR  = Path("../data/processed/images")
PRIVATE_DIR    = Path("../data/processed/images")
QUARANTINE_DIR = Path("../data/quarantine")
SPLIT_DIR      = Path("../data/split")
REPORTS_DIR    = Path("../reports")
MANIFEST_PATH  = Path("../logs/manifest.csv")
MAPPING_PATH   = Path("../logs/PRIVATE_filename_mapping.csv")
AUDIT_LOG      = Path("../logs/image_audit.jsonl")

# Supported extensions
SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

# Create all directories upfront
for d in [PROCESSED_DIR, QUARANTINE_DIR, REPORTS_DIR,
          SPLIT_DIR, MANIFEST_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ All directories initialized")

✓ All directories initialized


In [5]:
import json
from datetime import datetime

def log_audit(img_path, faces_found, method, strategy, status, note=""):
    
    record = {
        "timestamp"        : datetime.utcnow().isoformat(),
        "image"            : str(img_path),
        "faces_detected"   : faces_found,
        "detection_method" : method,
        "blur_strategy"    : strategy,
        "status"           : status,
        "note"             : note
    }
    AUDIT_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(AUDIT_LOG, "a") as f:
        f.write(json.dumps(record) + "\n")

# Quick test
log_audit("test.jpg", 0, "test", "landmark", "test_ok", note="fix3_check")
with open(AUDIT_LOG) as f:
    lines = [l.strip() for l in f if l.strip()]
last = json.loads(lines[-1])
print("✓ log_audit works:", last["timestamp"])

✓ log_audit works: 2026-03-12T05:08:20.426751


In [6]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import imagehash
import matplotlib.pyplot as plt
from pathlib import Path

In [7]:
import os

RAW_IMAGE_DIR = "../data/raw/skin_images"

for folder in ["train", "test"]:
    folder_path = os.path.join(RAW_IMAGE_DIR, folder)
    print(f"\nContents of {folder}:")
    for cls in os.listdir(folder_path):
        cls_path = os.path.join(folder_path, cls)
        print(f"  {cls}: {len(os.listdir(cls_path))} images")


Contents of train:
  acne: 586 images
  actinic_keratosis: 724 images
  benign_tumors: 1082 images
  bullous: 478 images
  candidiasis: 243 images
  drugeruption: 506 images
  eczema: 945 images
  infestations_bites: 508 images
  lichen: 535 images
  lupus: 302 images
  moles: 347 images
  psoriasis: 795 images
  rosacea: 253 images
  seborrh_keratoses: 445 images
  skincancer: 674 images
  sun_sunlight_damage: 310 images
  tinea: 893 images
  unknown_normal: 1622 images
  vascular_tumors: 533 images
  vasculitis: 453 images
  vitiligo: 674 images
  warts: 554 images

Contents of test:
  Acne: 63 images
  Actinic_Keratosis: 77 images
  Benign_tumors: 120 images
  Bullous: 48 images
  Candidiasis: 27 images
  DrugEruption: 47 images
  Eczema: 98 images
  Infestations_Bites: 56 images
  Lichen: 59 images
  Lupus: 32 images
  Moles: 35 images
  Psoriasis: 80 images
  Rosacea: 28 images
  Seborrh_Keratoses: 50 images
  SkinCancer: 76 images
  Sun_Sunlight_Damage: 32 images
  Tinea: 90 ima

In [8]:
from pathlib import Path
import os

RAW_IMAGE_DIR = Path("../data/raw/skin_images") 

for folder in ["train", "test"]:
    folder_path = RAW_IMAGE_DIR / folder
    if not folder_path.exists():
        continue
    for cls in os.listdir(folder_path):
        new_name = cls.lower().replace(" ", "_")
        old_path = folder_path / cls
        new_path = folder_path / new_name
        if old_path != new_path:
            os.rename(old_path, new_path)
            print(f"  Renamed: {cls} → {new_name}")
print("✓ Class folders normalized")

✓ Class folders normalized


DETECT CORRUPTED IMAGE

In [9]:
from PIL import Image
from tqdm import tqdm

corrupted_images = []

for folder in ["train", "test"]:
    folder_path = os.path.join(RAW_IMAGE_DIR, folder)
    
    for cls in os.listdir(folder_path):
        cls_path = os.path.join(folder_path, cls)
        for img_name in tqdm(os.listdir(cls_path), desc=f"Checking {folder}/{cls}"):
            img_path = os.path.join(cls_path, img_name)
            try:
                img = Image.open(img_path)
                img.verify()
                img = Image.open(img_path)
                img.load()  # fully load pixels to catch truncated images
            except:
                corrupted_images.append(img_path)

print(f"Detected {len(corrupted_images)} corrupted images")

Checking test/Warts: 100%|██████████| 61/61 [00:00<00:00, 315.69it/s]

Detected 0 corrupted images


QUARANTINE CORRUPTED IMAGES

In [10]:
import shutil

for img_path in corrupted_images:
    img_path = Path(img_path)
    dest = QUARANTINE_DIR / img_path.parent.name / img_path.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(img_path), dest)

print(f"Moved {len(corrupted_images)} corrupted images to quarantine.")

Moved 0 corrupted images to quarantine.


REMOVE DUPLICATE IMAGES

In [11]:
import imagehash
from PIL import Image

image_hashes = {}
duplicates = []
leakage_cases = []  # track cross-split duplicates separately

for folder in ["train", "test"]:
    folder_path = os.path.join(RAW_IMAGE_DIR, folder)
    for cls in os.listdir(folder_path):
        cls_path = os.path.join(folder_path, cls)
        for img_name in tqdm(os.listdir(cls_path), desc=f"Hashing {folder}/{cls}"):
            img_path = os.path.join(cls_path, img_name)
            try:
                img = Image.open(img_path).convert("RGB")
                img_hash = imagehash.phash(img)
                if img_hash in image_hashes:
                    existing_path = image_hashes[img_hash]
                    # Check if duplicate spans train and test
                    if ("train" in existing_path and "test" in img_path) or \
                       ("test" in existing_path and "train" in img_path):
                        leakage_cases.append((existing_path, img_path))
                    duplicates.append(img_path)
                else:
                    image_hashes[img_hash] = img_path
            except:
                continue

for dup_path in duplicates:
    shutil.move(dup_path, QUARANTINE_DIR)

print(f"Moved {len(duplicates)} duplicate images to quarantine.")
print(f"⚠️  {len(leakage_cases)} cross-split leakage cases detected and removed")

Hashing test/Warts: 100%|██████████| 61/61 [00:00<00:00, 208.41it/s]

Moved 0 duplicate images to quarantine.
⚠️  0 cross-split leakage cases detected and removed


STANDARDIZE IMAGE FORMAT (JPG, RGB)

In [12]:
PROCESSED_DIR = "../data/processed/images"
PRIVATE_DIR = "../data/private/anonymized_images"

import os
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(PRIVATE_DIR, exist_ok=True)

for folder in ["train", "test"]:
    folder_path = os.path.join(RAW_IMAGE_DIR, folder)
    processed_folder_path = os.path.join(PROCESSED_DIR, folder)
    private_folder_path = os.path.join(PRIVATE_DIR, folder)
    os.makedirs(processed_folder_path, exist_ok=True)
    os.makedirs(private_folder_path, exist_ok=True)
    
    for cls in os.listdir(folder_path):
        cls_path = os.path.join(folder_path, cls)
        processed_cls_path = os.path.join(processed_folder_path, cls)
        private_cls_path = os.path.join(private_folder_path, cls)
        os.makedirs(processed_cls_path, exist_ok=True)
        os.makedirs(private_cls_path, exist_ok=True)
        
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            try:
                img = Image.open(img_path).convert("RGB")
                save_path = os.path.join(processed_cls_path, os.path.splitext(img_name)[0] + ".jpg")
                img.save(save_path, "JPEG")
            except:
                continue

RESIZE IMAGES TO 224x224

In [13]:
from PIL import ImageOps

IMAGE_SIZE = (224, 224)

for folder in ["train", "test"]:
    processed_folder_path = os.path.join(PROCESSED_DIR, folder)
    for cls in os.listdir(processed_folder_path):
        cls_path = os.path.join(processed_folder_path, cls)
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            try:
                img = Image.open(img_path)
                img_resized = ImageOps.pad(img, IMAGE_SIZE, method=Image.LANCZOS, color=(0,0,0))
                img_resized.save(img_path)
            except Exception as e:
                print(f"Failed: {img_path} — {e}")



In [14]:
# Verify audit log path
print("AUDIT_LOG path :", AUDIT_LOG.resolve())
print("Folder exists  :", AUDIT_LOG.parent.exists())
print("File size      :", AUDIT_LOG.stat().st_size if AUDIT_LOG.exists() else 0, "bytes")

AUDIT_LOG path : C:\Users\User\Desktop\skin_disease_ai_system\skin_disease_ai_system\logs\image_audit.jsonl
Folder exists  : True
File size      : 43986507 bytes


FACE DETECTION AND ANONYMIZATION

In [15]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import json
from pathlib import Path
from datetime import datetime
import urllib.request
import os

# Disease Class Sets
FACIAL_DISEASE_CLASSES = {
    "acne", "rosacea", "lupus", "eczema",
    "seborrh_keratoses", "drugeruption", "lichen"
}

BODY_DISEASE_CLASSES = {
    "skincancer", "actinic_keratosis", "benign_tumors",
    "bullous", "candidiasis", "infestations_bites",
    "moles", "psoriasis", "sun_sunlight_damage",
    "tinea", "unknown_normal", "vascular_tumors", "vasculitis"
}

# Paths
PRIVATE_DIR = Path("../data/processed/images")   
AUDIT_LOG   = Path("../logs/image_audit.jsonl")
AUDIT_LOG.parent.mkdir(parents=True, exist_ok=True)

# Model files
MODELS_DIR            = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FACE_DETECTOR_MODEL   = str(MODELS_DIR / "blaze_face_short_range.tflite")
FACE_LANDMARKER_MODEL = str(MODELS_DIR / "face_landmarker.task")

# Load Haar Cascades
frontal_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
profile_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_profileface.xml"
)

# Download MediaPipe model files
if not os.path.exists(FACE_DETECTOR_MODEL):
    print("Downloading face detector model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_detector/"
        "blaze_face_short_range/float16/latest/blaze_face_short_range.tflite",
        FACE_DETECTOR_MODEL
    )
    print("  ✓ blaze_face_short_range.tflite downloaded")
else:
    print("  ✓ blaze_face_short_range.tflite already exists")

if not os.path.exists(FACE_LANDMARKER_MODEL):
    print("Downloading face landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
        "face_landmarker/float16/latest/face_landmarker.task",
        FACE_LANDMARKER_MODEL
    )
    print("  ✓ face_landmarker.task downloaded")
else:
    print("  ✓ face_landmarker.task already exists")

# Initialize MediaPipe Face Detector
face_detector_options = mp_vision.FaceDetectorOptions(
    base_options=mp_python.BaseOptions(model_asset_path=FACE_DETECTOR_MODEL),
    min_detection_confidence=0.5
)
mp_face = mp_vision.FaceDetector.create_from_options(face_detector_options)

# Initialize MediaPipe Face Landmarker 
face_landmarker_options = mp_vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=FACE_LANDMARKER_MODEL),
    num_faces=5,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5
)
mp_face_mesh = mp_vision.FaceLandmarker.create_from_options(face_landmarker_options)

print("✓ All detectors loaded successfully")

# Landmark Indices
LEFT_EYE_IDX    = [33, 7, 163, 144, 145, 153, 154, 155, 133,
                   246, 161, 160, 159, 158, 157, 173]
RIGHT_EYE_IDX   = [263, 249, 390, 373, 374, 380, 381, 382, 362,
                   466, 388, 387, 386, 385, 384, 398]
NOSE_BRIDGE_IDX = [6, 197, 195, 5, 4, 1, 19, 94, 2]

# Functions
def get_blur_strategy(cls_name: str) -> str:
    # Exact match first (handles CamelCase folder names)
    if cls_name in BODY_DISEASE_CLASSES:
        return "fullface"
    if cls_name in FACIAL_DISEASE_CLASSES:
        return "landmark"
    # Normalize fallback for any edge cases
    name = cls_name.lower().replace(" ", "_").replace("-", "_")
    for c in BODY_DISEASE_CLASSES:
        if c.lower() == name:
            return "fullface"
    for c in FACIAL_DISEASE_CLASSES:
        if c.lower() == name:
            return "landmark"
    print(f"    [WARN] Unknown class '{cls_name}' — defaulting to landmark (safe)")
    return "landmark"

def detect_faces(img_bgr):
    h, w   = img_bgr.shape[:2]
    boxes  = []
    method = "mediapipe"

    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    results  = mp_face.detect(mp_image)

    if results.detections:
        for det in results.detections:
            bb = det.bounding_box
            x  = max(0, bb.origin_x)
            y  = max(0, bb.origin_y)
            bw = bb.width
            bh = bb.height
            boxes.append((x, y, bw, bh))

    if not boxes:
        method = "haar_cascade"
        gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        frontal = frontal_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
        )
        boxes.extend([tuple(f) for f in frontal])
        profile_l = profile_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
        )
        boxes.extend([tuple(p) for p in profile_l])
        gray_flip = cv2.flip(gray, 1)
        profile_r = profile_cascade.detectMultiScale(
            gray_flip, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
        )
        for (x, y, bw, bh) in profile_r:
            boxes.append((w - x - bw, y, bw, bh))

    return boxes, method

def blur_landmarks_only(img_bgr):
    h, w     = img_bgr.shape[:2]
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result   = mp_face_mesh.detect(mp_image)
    blurred  = False

    if not result.face_landmarks:
        return img_bgr, False

    for face_landmarks in result.face_landmarks:
        lm = face_landmarks
        for idx_group in [LEFT_EYE_IDX, RIGHT_EYE_IDX, NOSE_BRIDGE_IDX]:
            pts = np.array([
                (int(lm[i].x * w), int(lm[i].y * h))
                for i in idx_group
            ], dtype=np.int32)
            x1 = max(0, pts[:, 0].min() - 15)
            y1 = max(0, pts[:, 1].min() - 15)
            x2 = min(w, pts[:, 0].max() + 15)
            y2 = min(h, pts[:, 1].max() + 15)
            if x2 <= x1 or y2 <= y1:
                continue
            region = img_bgr[y1:y2, x1:x2]
            k = max(31, ((x2 - x1) // 4) | 1)
            img_bgr[y1:y2, x1:x2] = cv2.GaussianBlur(region, (k, k), 0)
            blurred = True

    return img_bgr, blurred

def blur_full_face(img_bgr, faces):
    h, w = img_bgr.shape[:2]
    for (x, y, bw, bh) in faces:
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(w, x + bw), min(h, y + bh)
        if x2 <= x1 or y2 <= y1:
            continue
        region = img_bgr[y1:y2, x1:x2]
        k = max(51, (bw // 5) | 1)
        img_bgr[y1:y2, x1:x2] = cv2.GaussianBlur(region, (k, k), 0)
    return img_bgr

def process_image(img_path: Path, output_path: Path, cls_name: str):
    strategy = get_blur_strategy(cls_name)
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"    [SKIP] Cannot read: {img_path.name}")
        log_audit(img_path, 0, "none", strategy, "skipped",
                  note="cv2.imread returned None")
        return None

    faces_found = 0
    method      = "none"
    status      = "no_face_found"

    if strategy == "landmark":
        img, blurred = blur_landmarks_only(img)
        method       = "mediapipe_facemesh"
        faces_found  = 1 if blurred else 0
        status       = "landmark_blurred" if blurred else "no_landmarks_found"
    else:
        faces, method = detect_faces(img)
        faces_found   = len(faces)
        if faces:
            img    = blur_full_face(img, faces)
            status = "fullface_blurred"

    output_path.parent.mkdir(parents=True, exist_ok=True)
    success = cv2.imwrite(str(output_path), img)
    if not success:
        status = "write_failed"
        print(f"    [ERROR] Write failed: {output_path}")

    log_audit(img_path, faces_found, method, strategy, status)

    return {
        "original_name"    : img_path.name,
        "output_path"      : str(output_path),
        "class_name"       : cls_name,
        "blur_strategy"    : strategy,
        "detection_method" : method,
        "faces_detected"   : faces_found,
        "status"           : status
    }

# Verify everything is ready before running the loop
print(f"\n=== READY CHECK ===")
print(f"PRIVATE_DIR exists  : {PRIVATE_DIR.exists()}")
print(f"AUDIT_LOG path      : {AUDIT_LOG.resolve()}")
print(f"Face detector model : {os.path.exists(FACE_DETECTOR_MODEL)}")
print(f"Face landmarker     : {os.path.exists(FACE_LANDMARKER_MODEL)}")
img_count = len(list(PRIVATE_DIR.rglob("*.jpg")))
print(f"Images found        : {img_count}")

  ✓ blaze_face_short_range.tflite already exists
  ✓ face_landmarker.task already exists
✓ All detectors loaded successfully

=== READY CHECK ===
PRIVATE_DIR exists  : True
AUDIT_LOG path      : C:\Users\User\Desktop\skin_disease_ai_system\skin_disease_ai_system\logs\image_audit.jsonl
Face detector model : True
Face landmarker     : True
Images found        : 77282


PIXEL RE-ENCODING
FILENAME ANONYMIZATION
TIMESTAMP NORMALIZATION
MANIFEST CSV

In [16]:
from pathlib import Path
import pandas as pd
import uuid
import os
from datetime import datetime
import calendar
import hashlib
import io
from PIL import Image
import mediapipe as mp
import cv2
from tqdm import tqdm
import json

# Neutral UTC timestamp (timezone-safe)
neutral_timestamp = calendar.timegm(
    datetime(2000, 1, 1, 0, 0, 0).timetuple()
)

# Paths (fixed to match actual image location)
PRIVATE_DIR   = Path("../data/processed/images")         
MANIFEST_PATH = Path("../logs/manifest.csv")
MAPPING_PATH  = Path("../logs/PRIVATE_filename_mapping.csv")
AUDIT_LOG     = Path("../logs/image_audit.jsonl")

# Ensure directories exist
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
if not AUDIT_LOG.exists():
    AUDIT_LOG.touch()
    print(f"✓ Created empty audit log at {AUDIT_LOG.resolve()}")

# Verify before starting
print(f"PRIVATE_DIR   : {PRIVATE_DIR.resolve()}")
print(f"PRIVATE_DIR exists: {PRIVATE_DIR.exists()}")
print(f"Images found  : {len(list(PRIVATE_DIR.rglob('*.jpg')))}")

SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

manifest        = []
mapping_records = []
stats = {"total": 0, "processed": 0, "skipped": 0, "write_failed": 0}

# Utility Functions
def normalize_timestamps(path: Path):
    try:
        os.utime(path, (neutral_timestamp, neutral_timestamp))
    except Exception as e:
        print(f"    [WARN] Timestamp norm failed for {path.name}: {e}")

def pixel_reencode(img_bgr, output_path: Path):
    """Re-encode through PIL pixel buffer — zero metadata inheritance."""
    img_rgb = img_bgr[:, :, ::-1]      # BGR → RGB
    pil_img = Image.fromarray(img_rgb)
    if pil_img.mode != "RGB":
        pil_img = pil_img.convert("RGB")
    buffer = io.BytesIO()
    pil_img.save(buffer, format="JPEG", quality=95,
                 optimize=True, exif=b"", icc_profile=None)
    buffer.seek(0)
    output_path.write_bytes(buffer.read())

# Main Processing Loop
for split in ["train", "test"]:
    split_path = PRIVATE_DIR / split
    if not split_path.exists():
        print(f"[WARN] Split not found: {split_path}")
        continue

    class_list = sorted([
        c for c in os.listdir(split_path)
        if (split_path / c).is_dir()
    ])

    for cls_name in tqdm(class_list, desc=f"[{split.upper()}] Classes"):
        cls_path  = split_path / cls_name
        img_files = sorted([
            f for f in cls_path.iterdir()
            if f.suffix.lower() in SUPPORTED_EXT
        ])

        for img_file in tqdm(img_files, desc=f"  {cls_name}", leave=False):
            stats["total"] += 1

            # 1. Generate UUID filename
            anon_id      = str(uuid.uuid4())
            new_filename = f"{anon_id}.jpg"
            output_path  = cls_path / new_filename

            # 2. Face blur
            result = process_image(img_file, output_path, cls_name)
            if result is None:
                stats["skipped"] += 1
                continue

            # 3. Pixel re-encode
            raw = cv2.imread(str(output_path))
            if raw is not None:
                pixel_reencode(raw, output_path)
            else:
                stats["write_failed"] += 1
                print(f"    [ERROR] Re-encode failed — cannot read: {output_path.name}")
                continue

            # 4. Compute sha256_original BEFORE deleting the file
            sha256_orig = hashlib.sha256(img_file.read_bytes()).hexdigest()

            # 5. Normalize timestamps
            normalize_timestamps(img_file)
            normalize_timestamps(output_path)

            # 6. Remove original
            if output_path != img_file:
                try:
                    img_file.unlink()
                except Exception as e:
                    print(f"    [WARN] Could not remove {img_file.name}: {e}")

            # 7. Build manifest row
            result.pop("original_name", None)
            result["uuid_filename"]     = new_filename
            result["anonymized_id"]     = anon_id
            result["sha256_processed"]  = hashlib.sha256(
                    output_path.read_bytes()
                ).hexdigest()
            result["processing_date"]   = datetime.utcnow().strftime("%Y-%m-%d")
            result["mediapipe_version"] = mp.__version__
            manifest.append(result)
            stats["processed"] += 1

            # 8. Build private mapping row
            mapping_records.append({
                "uuid_filename"    : new_filename,
                "anonymized_id"    : anon_id,
                "original_filename": img_file.name,
                "original_path"    : str(img_file),
                "label"            : cls_name,
                "split"            : split,
                "sha256_original"  : sha256_orig
            })

# Normalize directory timestamps 
for split in ["train", "test"]:
    split_path = PRIVATE_DIR / split
    if not split_path.exists():
        continue
    for cls_name in os.listdir(split_path):
        normalize_timestamps(split_path / cls_name)
    normalize_timestamps(split_path)
normalize_timestamps(PRIVATE_DIR)

# Save manifest CSV
df_manifest = pd.DataFrame(manifest)
df_manifest.to_csv(MANIFEST_PATH, index=False)

# Save private mapping
df_mapping = pd.DataFrame(mapping_records)
df_mapping.to_csv(MAPPING_PATH, index=False)

print(f"""
╔══════════════════════════════════════════════════════════╗
  PIPELINE COMPLETE
╠══════════════════════════════════════════════════════════╣
  Total images     : {stats['total']}
  Processed        : {stats['processed']}
  Skipped          : {stats['skipped']}
  Write failures   : {stats['write_failed']}
╠══════════════════════════════════════════════════════════╣
  Manifest         : {MANIFEST_PATH.resolve()}
  Private mapping  : {MAPPING_PATH.resolve()}
  Audit log        : {AUDIT_LOG.resolve()}
╚══════════════════════════════════════════════════════════╝
""")

print(f"⚠  Keep PRIVATE_filename_mapping.csv out of git and shared folders")

PRIVATE_DIR   : C:\Users\User\Desktop\skin_disease_ai_system\skin_disease_ai_system\data\processed\images
PRIVATE_DIR exists: True
Images found  : 77282


[TRAIN] Classes:   5%|▍         | 1/22 [03:45<1:18:49, 225.22s/it]


PermissionError: [Errno 13] Permission denied: '..\\logs\\image_audit.jsonl'

QUICK CHECK

1. Loop Produced Results

In [ ]:
print("=== QUICK SANITY CHECK ===")
print(f"Manifest rows    : {len(manifest)}")
print(f"Mapping rows     : {len(mapping_records)}")
print(f"Stats            : {stats}")

2. Audit Log Was Written

In [ ]:
import json
from pathlib import Path

AUDIT_LOG = Path("../logs/image_audit.jsonl")
print(f"Audit log size   : {AUDIT_LOG.stat().st_size} bytes")

with open(AUDIT_LOG) as f:
    records = [json.loads(l) for l in f if l.strip()]

import pandas as pd
df_audit = pd.DataFrame(records)
print(f"Audit records    : {len(records)}")
print(f"\nStatus breakdown:")
print(df_audit["status"].value_counts())
print(f"\nBlur strategy:")
print(df_audit["blur_strategy"].value_counts())

3. Manifest and Mapping CSVs Are Valid

In [ ]:
MANIFEST_PATH = Path("../logs/manifest.csv")
MAPPING_PATH  = Path("../logs/PRIVATE_filename_mapping.csv")

df_manifest = pd.read_csv(MANIFEST_PATH)
df_mapping  = pd.read_csv(MAPPING_PATH)

print(f"Manifest rows    : {len(df_manifest)}")
print(f"Mapping rows     : {len(df_mapping)}")
print(f"Manifest columns : {list(df_manifest.columns)}")

# PII check
if "original_name" in df_manifest.columns:
    print("❌ original_name column still present — PII not removed")
else:
    print("✅ No PII in manifest")

# Row count match
if len(df_manifest) == len(df_mapping):
    print("✅ Manifest and mapping row counts match")
else:
    print(f"❌ Row count mismatch — manifest:{len(df_manifest)} mapping:{len(df_mapping)}")

4. All Filenames Are UUID

In [ ]:
import re
from pathlib import Path

PRIVATE_DIR  = Path("../data/processed/images")
uuid_pattern = re.compile(
    r'^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}\.jpg$'
)

non_uuid = [
    f for f in PRIVATE_DIR.rglob("*.jpg")
    if not uuid_pattern.match(f.name)
]

if non_uuid:
    print(f"❌ {len(non_uuid)} non-UUID filenames found:")
    for f in non_uuid[:5]:
        print(f"   {f.relative_to(PRIVATE_DIR)}")
else:
    print(f"✅ All filenames are UUID-anonymized")

DATASET REPORT GENERATION

In [ ]:
import os
import json
import math
import base64
import datetime
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm import tqdm

# Paths
PRIVATE_DIR   = Path("../data/processed/images")
REPORTS_DIR   = Path("../reports")
AUDIT_LOG     = Path("../logs/image_audit.jsonl")
MANIFEST_PATH = Path("../logs/manifest.csv")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

print("Generating dataset report...")

1. Dataset Overview

In [ ]:
class_counts   = {"train": {}, "test": {}}
all_img_paths  = {"train": [], "test": []}

for split in ["train", "test"]:
    split_path = PRIVATE_DIR / split
    if not split_path.exists():
        continue
    for cls in sorted(split_path.iterdir()):
        if not cls.is_dir():
            continue
        imgs = [f for f in cls.iterdir() if f.suffix.lower() in SUPPORTED_EXT]
        class_counts[split][cls.name] = len(imgs)
        all_img_paths[split].extend(imgs)

total_train   = sum(class_counts["train"].values())
total_test    = sum(class_counts["test"].values())
total_images  = total_train + total_test
all_classes   = sorted(set(
    list(class_counts["train"].keys()) +
    list(class_counts["test"].keys())
))

print(f"  ✓ Section 1 — Overview: {total_images} images, {len(all_classes)} classes")

2. Class Distribution & Imbalance Analysis

In [ ]:
train_counts  = [class_counts["train"].get(c, 0) for c in all_classes]
test_counts   = [class_counts["test"].get(c, 0)  for c in all_classes]
total_per_cls = [t + v for t, v in zip(train_counts, test_counts)]

max_cls = all_classes[np.argmax(total_per_cls)]
min_cls = all_classes[np.argmin(total_per_cls)]
imbalance_ratio = max(total_per_cls) / max(min(total_per_cls), 1)

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(all_classes))
bars_train = ax.bar(x - 0.2, train_counts, 0.4,
                    label="Train", color="#2196F3", alpha=0.85)
bars_test  = ax.bar(x + 0.2, test_counts,  0.4,
                    label="Test",  color="#FF5722", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(all_classes, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Image Count")
ax.set_title("Class Distribution — Train vs Test", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
dist_chart_path = REPORTS_DIR / "class_distribution.png"
plt.savefig(dist_chart_path, dpi=150, bbox_inches="tight")
plt.close()

print(f"  ✓ Section 2 — Class distribution chart saved")

3. Privacy-Preserving Summary

In [ ]:
audit_records = []
if AUDIT_LOG.exists() and AUDIT_LOG.stat().st_size > 0:
    with open(AUDIT_LOG) as f:
        audit_records = [json.loads(l) for l in f if l.strip()]

df_audit = pd.DataFrame(audit_records) if audit_records else pd.DataFrame()

if not df_audit.empty:
    status_counts   = df_audit["status"].value_counts().to_dict()
    method_counts   = df_audit["detection_method"].value_counts().to_dict()
    strategy_counts = df_audit["blur_strategy"].value_counts().to_dict()
    faces_detected  = int(df_audit["faces_detected"].sum())
else:
    status_counts   = {}
    method_counts   = {}
    strategy_counts = {}
    faces_detected  = 0

print(f"  ✓ Section 3 — Privacy summary: {len(audit_records)} audit records")

4. Pixel Statistics

In [ ]:
print("  Computing pixel statistics (train set)...")
r_vals, g_vals, b_vals = [], [], []
sample_paths = all_img_paths["train"]

# Sample max 2000 images for speed
if len(sample_paths) > 2000:
    rng     = np.random.default_rng(42)
    indices = rng.choice(len(sample_paths), 2000, replace=False)
    sample_paths = [sample_paths[i] for i in indices]

for img_path in tqdm(sample_paths, desc="  Pixel stats", leave=False):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    r_vals.append(img_rgb[:, :, 0].mean())
    g_vals.append(img_rgb[:, :, 1].mean())
    b_vals.append(img_rgb[:, :, 2].mean())

pixel_stats = {
    "mean": {
        "R": round(float(np.mean(r_vals)), 4),
        "G": round(float(np.mean(g_vals)), 4),
        "B": round(float(np.mean(b_vals)), 4)
    },
    "std": {
        "R": round(float(np.std(r_vals)), 4),
        "G": round(float(np.std(g_vals)), 4),
        "B": round(float(np.std(b_vals)), 4)
    },
    "samples_used": len(r_vals)
}

print(f"  ✓ Section 4 — Pixel stats: mean R={pixel_stats['mean']['R']} "
      f"G={pixel_stats['mean']['G']} B={pixel_stats['mean']['B']}")

In [ ]:
import numpy as np
from pathlib import Path

print("===COMPUTING NORMALIZATION STATS ===")

norm_stats = {
    "mean_list" : [
        pixel_stats["mean"]["R"],
        pixel_stats["mean"]["G"],
        pixel_stats["mean"]["B"]
    ],
    "std_list"  : [
        pixel_stats["std"]["R"],
        pixel_stats["std"]["G"],
        pixel_stats["std"]["B"]
    ],
    "samples_used" : pixel_stats["samples_used"],
    "note"         : "Computed from train set sample (max 2000 images)"
}

norm_path = Path("../reports/normalization_stats.json")
with open(norm_path, "w") as f:
    json.dump(norm_stats, f, indent=2)

print(f"  Mean : R={norm_stats['mean_list'][0]}  "
      f"G={norm_stats['mean_list'][1]}  "
      f"B={norm_stats['mean_list'][2]}")
print(f"  Std  : R={norm_stats['std_list'][0]}  "
      f"G={norm_stats['std_list'][1]}  "
      f"B={norm_stats['std_list'][2]}")
print(f"✅ normalization_stats.json saved → {norm_path.resolve()}")

5. Sample Image Grid (3 per class)

In [ ]:
print("  Building sample image grid...")
n_classes   = len(all_classes)
n_cols      = 3
fig_rows    = n_classes
fig, axes   = plt.subplots(fig_rows, n_cols,
                            figsize=(n_cols * 3, fig_rows * 3))

for row_idx, cls_name in enumerate(all_classes):
    cls_path = PRIVATE_DIR / "train" / cls_name
    if not cls_path.exists():
        cls_path = PRIVATE_DIR / "test" / cls_name

    imgs = sorted([
        f for f in cls_path.iterdir()
        if f.suffix.lower() in SUPPORTED_EXT
    ])[:n_cols]

    for col_idx in range(n_cols):
        ax = axes[row_idx][col_idx] if n_classes > 1 else axes[col_idx]
        if col_idx < len(imgs):
            img = cv2.imread(str(imgs[col_idx]))
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_rgb)
        ax.axis("off")
        if col_idx == 0:
            ax.set_title(cls_name, fontsize=8, fontweight="bold",
                         loc="left", pad=2)

plt.suptitle("Sample Images Per Class (3 per class — Train set)",
             fontsize=12, fontweight="bold", y=1.001)
plt.tight_layout()
grid_path = REPORTS_DIR / "sample_image_grid.png"
plt.savefig(grid_path, dpi=80, bbox_inches="tight")
plt.close()

print(f"  ✓ Section 5 — Sample grid saved")

6. JSON Report (For NLP Integration)

In [ ]:
report_json = {
    "report_generated"  : datetime.datetime.utcnow().isoformat(),
    "overview": {
        "total_images"  : total_images,
        "total_classes" : len(all_classes),
        "train_images"  : total_train,
        "test_images"   : total_test,
        "train_ratio"   : round(total_train / max(total_images, 1), 3),
        "test_ratio"    : round(total_test  / max(total_images, 1), 3),
        "classes"       : all_classes
    },
    "class_distribution": {
        "train"             : class_counts["train"],
        "test"              : class_counts["test"],
        "most_common_class" : max_cls,
        "least_common_class": min_cls,
        "imbalance_ratio"   : round(imbalance_ratio, 2),
        "imbalance_warning" : imbalance_ratio > 5
    },
    "privacy_summary": {
        "total_audit_records": len(audit_records),
        "faces_detected_total": faces_detected,
        "status_breakdown"    : status_counts,
        "detection_method"    : method_counts,
        "blur_strategy"       : strategy_counts
    },
    "pixel_statistics": pixel_stats,
    "preprocessing_steps": [
        "Corrupted image detection and removal",
        "Duplicate image removal (pHash)",
        "Format standardization to JPG",
        "Resize to 224x224 (LANCZOS)",
        "EXIF metadata removal",
        "Face detection and anonymization (MediaPipe + Haar)",
        "Landmark-only blur for facial disease classes",
        "Full face blur for body-site disease classes",
        "Pixel re-encoding through PIL buffer",
        "Filename anonymization (UUID)",
        "File timestamp normalization (2000-01-01)"
    ]
}

json_report_path = REPORTS_DIR / "dataset_report.json"
with open(json_report_path, "w") as f:
    json.dump(report_json, f, indent=2)

print(f"  ✓ Section 6 — JSON report saved")

CLASS DISTRIBUTION OVERVIEW

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PRIVATE_DIR = Path("../data/processed/images")

print("=== CLASS DISTRIBUTION ===")
class_data = []
for split in ["train", "test"]:
    split_path = PRIVATE_DIR / split
    if not split_path.exists():
        continue
    for cls in sorted(split_path.iterdir()):
        if not cls.is_dir():
            continue
        count = len([f for f in cls.iterdir()
                     if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        class_data.append({
            "class": cls.name,
            "split": split,
            "count": count
        })

df = pd.DataFrame(class_data)
pivot = df.pivot(index="class", columns="split", values="count").fillna(0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)

print(pivot.to_string())
print(f"\nMax class : {pivot['total'].idxmax()} ({int(pivot['total'].max())} images)")
print(f"Min class : {pivot['total'].idxmin()} ({int(pivot['total'].min())} images)")
print(f"Imbalance ratio : {pivot['total'].max() / pivot['total'].min():.2f}x")

In [ ]:
pip install scikit-learn --break-system-packages

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import defaultdict

# Paths
PRIVATE_DIR   = Path("../data/processed/images")
SPLIT_DIR     = Path("../data/split")         
MANIFEST_PATH = Path("../logs/manifest.csv")
REPORTS_DIR   = Path("../reports")

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXT = {".jpg", ".jpeg", ".png"}
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15
RANDOM_SEED   = 42

# Step 1: Collect all images and labels
print("=== COLLECTING ALL IMAGES ===")
all_images = []   # list of (path, class_name)

for split in ["train", "test"]:
    split_path = PRIVATE_DIR / split
    if not split_path.exists():
        continue
    for cls in sorted(split_path.iterdir()):
        if not cls.is_dir():
            continue
        for img in cls.iterdir():
            if img.suffix.lower() in SUPPORTED_EXT:
                all_images.append((img, cls.name))

total = len(all_images)
print(f"Total images collected : {total}")

# Step 2: Separate by class for stratified split
class_to_images = defaultdict(list)
for img_path, cls_name in all_images:
    class_to_images[cls_name].append(img_path)

print(f"Total classes          : {len(class_to_images)}")

# Step 3: Stratified split per class
print("\n=== SPLITTING PER CLASS ===")
split_records = []
skipped_classes = []

for cls_name, img_list in sorted(class_to_images.items()):
    n = len(img_list)

    # Minimum images needed for 3-way stratified split
    if n < 10:
        print(f"  [WARN] {cls_name}: only {n} images — assigning all to train")
        for img in img_list:
            split_records.append((img, cls_name, "train"))
        skipped_classes.append(cls_name)
        continue

    # First split: separate test set
    train_val, test = train_test_split(
        img_list,
        test_size=TEST_RATIO,
        random_state=RANDOM_SEED
    )

    # Second split: separate val from train
    val_size_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train, val = train_test_split(
        train_val,
        test_size=val_size_adjusted,
        random_state=RANDOM_SEED
    )

    for img in train : split_records.append((img, cls_name, "train"))
    for img in val   : split_records.append((img, cls_name, "val"))
    for img in test  : split_records.append((img, cls_name, "test"))

    print(f"  {cls_name:30s} → train:{len(train):4d} | "
          f"val:{len(val):4d} | test:{len(test):4d}")

# Step 4: Copy images to new split folders 
print("\n=== COPYING IMAGES TO SPLIT FOLDERS ===")
stats = {"train": 0, "val": 0, "test": 0}

for img_path, cls_name, split in split_records:
    dest_dir = SPLIT_DIR / split / cls_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / img_path.name

    if not dest_path.exists():
        shutil.copy2(str(img_path), str(dest_path))

    stats[split] += 1

print(f"  Train : {stats['train']:,} images  ({stats['train']/total*100:.1f}%)")
print(f"  Val   : {stats['val']:,} images  ({stats['val']/total*100:.1f}%)")
print(f"  Test  : {stats['test']:,} images  ({stats['test']/total*100:.1f}%)")

# Step 5: Verify stratification
print("\n=== STRATIFICATION VERIFICATION ===")

df_records = pd.DataFrame(split_records, columns=["path", "class", "split"])
df_verify = (
    df_records.groupby(["class", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["train", "val", "test"], fill_value=0)
)
df_verify["total"] = df_verify.sum(axis=1)
df_verify["train%"] = (df_verify["train"] / df_verify["total"] * 100).round(1)
df_verify["val%"]   = (df_verify["val"]   / df_verify["total"] * 100).round(1)
df_verify["test%"]  = (df_verify["test"]  / df_verify["total"] * 100).round(1)
df_verify = df_verify.reset_index()

print(df_verify.to_string(index=False))

# Step 6: Visualize new distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart — counts per class per split
x      = np.arange(len(df_verify))
width  = 0.25
ax     = axes[0]
ax.bar(x - width, df_verify["train"], width, label="Train", color="#2196F3", alpha=0.85)
ax.bar(x,         df_verify["val"],   width, label="Val",   color="#4CAF50", alpha=0.85)
ax.bar(x + width, df_verify["test"],  width, label="Test",  color="#FF5722", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_verify["class"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Image Count")
ax.set_title("Class Distribution After Split", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Pie chart — overall split ratio
ax2 = axes[1]
ax2.pie(
    [stats["train"], stats["val"], stats["test"]],
    labels=[
        f"Train\n{stats['train']:,} ({stats['train']/total*100:.1f}%)",
        f"Val\n{stats['val']:,} ({stats['val']/total*100:.1f}%)",
        f"Test\n{stats['test']:,} ({stats['test']/total*100:.1f}%)"
    ],
    colors=["#2196F3", "#4CAF50", "#FF5722"],
    autopct="%1.1f%%",
    startangle=90
)
ax2.set_title("Overall Split Ratio", fontweight="bold")

plt.tight_layout()
split_chart_path = REPORTS_DIR / "split_distribution.png"
plt.savefig(split_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✓ Split chart saved → {split_chart_path}")

# Step 7: Save updated manifest
df_manifest = pd.DataFrame([
    {
        "image_path" : str(SPLIT_DIR / s / c / p.name),
        "label"      : c,
        "split"      : s,
        "filename"   : p.name
    }
    for p, c, s in split_records
])
df_manifest.to_csv(MANIFEST_PATH, index=False)
print(f"✓ Manifest updated  → {MANIFEST_PATH}")

# Final Summary
print(f"""
╔══════════════════════════════════════════════════════════╗
  STRATIFIED SPLIT COMPLETE
╠══════════════════════════════════════════════════════════╣
  Total images   : {total:,}
  Train          : {stats['train']:,}  ({stats['train']/total*100:.1f}%)
  Val            : {stats['val']:,}   ({stats['val']/total*100:.1f}%)
  Test           : {stats['test']:,}   ({stats['test']/total*100:.1f}%)
  Classes        : {len(class_to_images)}
  Output folder  : {SPLIT_DIR.resolve()}
╚══════════════════════════════════════════════════════════╝
""")

if skipped_classes:
    print(f"⚠️  Classes with < 10 images (all assigned to train):")
    for c in skipped_classes:
        print(f"   {c}")

In [ ]:
df_full     = pd.read_csv(MANIFEST_PATH)
df_privacy  = pd.read_csv(MAPPING_PATH)[
    ["uuid_filename", "sha256_original", "anonymized_id"]
]
df_merged = df_full.merge(
    df_privacy,
    left_on  = "filename",
    right_on = "uuid_filename",
    how      = "left"
).drop(columns=["uuid_filename"])
df_merged.to_csv(MANIFEST_PATH, index=False)
print(f"✓ Manifest merged with privacy fields → {MANIFEST_PATH}")
print(f"  Columns : {list(df_merged.columns)}")
print(f"  Rows    : {len(df_merged):,}")

SPLIT VERIFICATION

In [ ]:
from pathlib import Path

SPLIT_DIR = Path("../data/split")

print("=== SPLIT VERIFICATION ===")
for split in ["train", "val", "test"]:
    split_path = SPLIT_DIR / split
    if not split_path.exists():
        print(f"❌ {split} folder missing")
        continue
    total = len(list(split_path.rglob("*.jpg")))
    classes = len([c for c in split_path.iterdir() if c.is_dir()])
    print(f"{split:6s} → {total:5,} images | {classes} classes")

CLASS IMBALANCE HANDLING

In [ ]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from pathlib import Path
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import random
import shutil

# Paths
SPLIT_DIR    = Path("../data/split")
REPORTS_DIR  = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXT = {".jpg", ".jpeg", ".png"}
RANDOM_SEED   = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Thresholds
MINORITY_THRESHOLD = 500    # classes below this get augmented
MAJORITY_THRESHOLD = 1000   # classes above this get capped
TARGET_MINORITY    = 650    # augment minority classes up to this count

1. Compute Class Weights

In [ ]:
print("COMPUTING CLASS WEIGHTS")

train_path   = SPLIT_DIR / "train"
class_names  = sorted([c.name for c in train_path.iterdir() if c.is_dir()])
class_counts = {}

for cls in class_names:
    cls_path = train_path / cls
    count = len([f for f in cls_path.iterdir()
                 if f.suffix.lower() in SUPPORTED_EXT])
    class_counts[cls] = count

# Build label array for sklearn
labels = []
for i, cls in enumerate(class_names):
    labels.extend([i] * class_counts[cls])
labels = np.array(labels)

# Compute balanced class weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=labels
)

class_weights = {cls: round(float(w), 4)
                 for cls, w in zip(class_names, weights)}

print(f"  {'Class':<30} {'Count':>6}  {'Weight':>8}")
print(f"  {'-'*48}")
for cls in class_names:
    print(f"  {cls:<30} {class_counts[cls]:>6}  {class_weights[cls]:>8.4f}")

# Save class weights to JSON for model training
weights_path = REPORTS_DIR / "class_weights.json"
with open(weights_path, "w") as f:
    json.dump({
        "class_names"  : class_names,
        "class_weights": class_weights,
        "weight_array" : [class_weights[c] for c in class_names]
    }, f, indent=2)
print(f"\n✓ Class weights saved → {weights_path}")

original_counts = class_counts.copy()

2. Augmentation on Minority Classes

In [ ]:
print("AUGMENTING MINORITY CLASSES")

# Augmentation pipeline — skin disease appropriate transforms
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),
    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=20,
        val_shift_limit=10,
        p=0.4
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(var_limit=(5, 20), p=0.2),
    A.RandomResizedCrop(
        size=(224, 224),
        scale=(0.85, 1.0),
        p=0.3
    ),
])

minority_classes = {
    cls: count for cls, count in class_counts.items()
    if count < MINORITY_THRESHOLD
}

print(f"  Minority classes (< {MINORITY_THRESHOLD} images): "
      f"{len(minority_classes)}")

aug_stats = {}
for cls_name, original_count in tqdm(
        minority_classes.items(), desc="  Augmenting"):

    cls_path    = train_path / cls_name
    img_files   = [f for f in cls_path.iterdir()
                   if f.suffix.lower() in SUPPORTED_EXT]
    needed      = TARGET_MINORITY - original_count
    aug_count   = 0

    if needed <= 0:
        continue

    # Cycle through images and augment until target reached
    for i in range(needed):
        src = random.choice(img_files)
        img      = cv2.imread(str(src))
        if img is None:
            continue

        img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        aug_img  = augment(image=img_rgb)["image"]
        aug_bgr  = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)

        # Save with aug_ prefix so it's identifiable
        aug_name = f"aug_{i:04d}_{src.stem}.jpg"
        aug_path = cls_path / aug_name
        cv2.imwrite(str(aug_path), aug_bgr)
        aug_count += 1

    aug_stats[cls_name] = {
        "before": original_count,
        "added" : aug_count,
        "after" : original_count + aug_count
    }
    print(f"    {cls_name:<30} {original_count:>4} → "
          f"{original_count + aug_count:>4} (+{aug_count})")
    
# Refresh counts after augmentation 
print("\nRefreshing class counts after augmentation...")
for cls in class_names:
    cls_path = train_path / cls
    class_counts[cls] = len([
        f for f in cls_path.iterdir()
        if f.suffix.lower() in SUPPORTED_EXT
    ])
print("✓ Counts refreshed")

3. Cap Majority Classes

In [ ]:
old_max   = max(class_counts.values())
old_min   = min(class_counts.values())
old_ratio = round(old_max / max(old_min, 1), 2)

print(f"CAPPING MAJORITY CLASSES (> {MAJORITY_THRESHOLD})")

cap_stats = {}
for cls_name, count in class_counts.items():
    if count <= MAJORITY_THRESHOLD:
        continue

    cls_path  = train_path / cls_name
    img_files = [f for f in cls_path.iterdir()
                 if f.suffix.lower() in SUPPORTED_EXT
                 and not f.name.startswith("aug_")]  
    
    # Randomly select images to remove
    excess   = count - MAJORITY_THRESHOLD
    to_remove = random.sample(img_files, min(excess, len(img_files)))

    for f in to_remove:
        f.unlink()

    cap_stats[cls_name] = {
        "before" : count,
        "removed": len(to_remove),
        "after"  : count - len(to_remove)
    }
    print(f"    {cls_name:<30} {count:>5} → "
          f"{count - len(to_remove):>5} (-{len(to_remove)})")

if not cap_stats:
    print("    No classes exceeded the majority threshold — nothing capped")

In [ ]:
# Sanity check: no aug_ files in val or test
print("\n=== AUG LEAK CHECK ===")
for split in ["val", "test"]:
    aug_files = list((SPLIT_DIR / split).rglob("aug_*.jpg"))
    if len(aug_files) == 0:
        print(f"  ✅ {split} — clean, no aug_ files")
    else:
        print(f"  ⚠️  {split} — {len(aug_files)} aug_ files found!")

4. Verify Final Distribution

In [ ]:
old_max   = max(original_counts.values())
old_min   = min(original_counts.values())
old_ratio = round(old_max / max(old_min, 1), 2)

print("FINAL DISTRIBUTION")

final_counts = {}
for cls in class_names:
    cls_path = train_path / cls
    final_counts[cls] = len([
        f for f in cls_path.iterdir()
        if f.suffix.lower() in SUPPORTED_EXT
    ])

df_final = pd.DataFrame({
    "class"   : list(final_counts.keys()),
    "before"  : [original_counts[c] for c in final_counts.keys()],
    "after"   : list(final_counts.values())
})
df_final["change"] = df_final["after"] - df_final["before"]
df_final = df_final.sort_values("after", ascending=False)

print(df_final.to_string(index=False))

new_max   = df_final["after"].max()
new_min   = df_final["after"].min()
new_ratio = round(new_max / max(new_min, 1), 2)
print(f"  Old imbalance ratio : {old_ratio}x")
print(f"  New imbalance ratio : {new_ratio}x")

# Visualization: Before vs After
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

x     = np.arange(len(df_final))
width = 0.4

axes[0].bar(x - width/2, df_final["before"], width,
            label="Before", color="#FF5722", alpha=0.8)
axes[0].bar(x + width/2, df_final["after"],  width,
            label="After",  color="#2196F3", alpha=0.8)
axes[0].axhline(y=MINORITY_THRESHOLD, color="green",
                linestyle="--", alpha=0.7, label=f"Min target ({MINORITY_THRESHOLD})")
axes[0].axhline(y=MAJORITY_THRESHOLD, color="red",
                linestyle="--", alpha=0.7, label=f"Max cap ({MAJORITY_THRESHOLD})")
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_final["class"], rotation=45,
                         ha="right", fontsize=8)
axes[0].set_ylabel("Image Count")
axes[0].set_title("Before vs After Imbalance Handling", fontweight="bold")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Imbalance ratio comparison
axes[1].bar(["Before", "After"], [old_ratio, new_ratio],
            color=["#FF5722", "#2196F3"], alpha=0.85, width=0.4)
axes[1].set_ylabel("Imbalance Ratio (x)")
axes[1].set_title("Imbalance Ratio Improvement", fontweight="bold")
for i, v in enumerate([old_ratio, new_ratio]):
    axes[1].text(i, v + 0.5, f"{v}x", ha="center", fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
imbalance_chart = REPORTS_DIR / "imbalance_handling.png"
plt.savefig(imbalance_chart, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✓ Chart saved → {imbalance_chart}")


In [ ]:
from pathlib import Path
import json

MANIFEST_PATH = Path("../logs/manifest.csv")
SPLIT_DIR     = Path("../data/split")
REPORTS_DIR   = Path("../reports")

# Load norm_stats if not already in memory
try:
    norm_stats
except NameError:
    with open(REPORTS_DIR / "normalization_stats.json") as f:
        norm_stats = json.load(f)

# Load class_weights if not already in memory
try:
    class_weights
except NameError:
    with open(REPORTS_DIR / "class_weights.json") as f:
        cw_data       = json.load(f)
        class_names   = cw_data["class_names"]
        class_weights = cw_data["class_weights"]

config = {
    "class_names"   : class_names,
    "num_classes"   : len(class_names),
    "image_size"    : [224, 224],
    "splits"        : ["train", "val", "test"],
    "split_dir"     : str(SPLIT_DIR.resolve()),
    "manifest_path" : str(MANIFEST_PATH.resolve()),
    "class_weights" : class_weights,
    "norm_mean"     : norm_stats["mean_list"],
    "norm_std"      : norm_stats["std_list"]
}

config_path = Path("../config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ config.json saved → {config_path.resolve()}")
print(f"   Classes : {config['num_classes']}")
print(f"   Mean    : {config['norm_mean']}")
print(f"   Std     : {config['norm_std']}")

In [ ]:
from pathlib import Path
import pandas as pd

SPLIT_DIR = Path("../data/split")

print("=== FINAL DISTRIBUTION CHECK ===")
data = []
for split in ["train", "val", "test"]:
    split_path = SPLIT_DIR / split
    if not split_path.exists():
        continue
    for cls in sorted(split_path.iterdir()):
        if not cls.is_dir():
            continue
        count = len([f for f in cls.iterdir()
                     if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        data.append({"class": cls.name, "split": split, "count": count})

df = pd.DataFrame(data)
pivot = df.pivot(index="class", columns="split", values="count").fillna(0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("train", ascending=False)

print(pivot.to_string())
print(f"\nTrain total  : {int(pivot['train'].sum()):,}")
print(f"Val total    : {int(pivot['val'].sum()):,}")
print(f"Test total   : {int(pivot['test'].sum()):,}")
print(f"Grand total  : {int(pivot['total'].sum()):,}")

train_max   = pivot["train"].max()
train_min   = pivot["train"].min()
print(f"\nNew imbalance ratio : {train_max/max(train_min,1):.2f}x")

DATA NORMALIZATION

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm

# Paths
SPLIT_DIR   = Path("../data/split")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXT = {".jpg", ".jpeg", ".png"}
SAMPLE_LIMIT  = 3000

# Collect all training images
train_path  = SPLIT_DIR / "train"
all_imgs    = [
    f for f in train_path.rglob("*")
    if f.suffix.lower() in SUPPORTED_EXT
]

# Random sample for speed if dataset is large
if len(all_imgs) > SAMPLE_LIMIT:
    rng      = np.random.default_rng(42)
    indices  = rng.choice(len(all_imgs), SAMPLE_LIMIT, replace=False)
    all_imgs = [all_imgs[i] for i in indices]

print(f"Computing normalization stats from {len(all_imgs):,} training images...")

# Compute per-channel mean and std
r_sum = g_sum = b_sum = 0.0
r_sq  = g_sq  = b_sq  = 0.0
pixel_count = 0
skipped     = 0

for img_path in tqdm(all_imgs, desc="  Computing mean"):
    img = cv2.imread(str(img_path))
    if img is None:
        skipped += 1
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    r_sum  += img_rgb[:, :, 0].sum()
    g_sum  += img_rgb[:, :, 1].sum()
    b_sum  += img_rgb[:, :, 2].sum()
    r_sq   += (img_rgb[:, :, 0] ** 2).sum()
    g_sq   += (img_rgb[:, :, 1] ** 2).sum()
    b_sq   += (img_rgb[:, :, 2] ** 2).sum()
    pixel_count += img_rgb.shape[0] * img_rgb.shape[1]

# Mean
mean_r = r_sum / pixel_count
mean_g = g_sum / pixel_count
mean_b = b_sum / pixel_count

# Std  (E[X²] - E[X]²)
std_r = np.sqrt(r_sq / pixel_count - mean_r ** 2)
std_g = np.sqrt(g_sq / pixel_count - mean_g ** 2)
std_b = np.sqrt(b_sq / pixel_count - mean_b ** 2)

norm_stats = {
    "mean": {
        "R": round(float(mean_r), 4),
        "G": round(float(mean_g), 4),
        "B": round(float(mean_b), 4)
    },
    "std": {
        "R": round(float(std_r), 4),
        "G": round(float(std_g), 4),
        "B": round(float(std_b), 4)
    },
    "mean_list": [round(float(mean_r), 4),
                  round(float(mean_g), 4),
                  round(float(mean_b), 4)],
    "std_list" : [round(float(std_r), 4),
                  round(float(std_g), 4),
                  round(float(std_b), 4)],
    "samples_used"  : len(all_imgs) - skipped,
    "pixels_counted": pixel_count,
    "skipped"       : skipped,
    "note"          : "Computed from training set only. Use mean_list and std_list directly in transforms."
}

# Save to JSON
norm_path = REPORTS_DIR / "normalization_stats.json"
with open(norm_path, "w") as f:
    json.dump(norm_stats, f, indent=2)

# Compare with ImageNet stats
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

print(f"""
╔══════════════════════════════════════════════════════════╗
  NORMALIZATION STATS
╠══════════════════════════════════════════════════════════╣
  Your dataset mean  (R, G, B) : {norm_stats['mean_list']}
  Your dataset std   (R, G, B) : {norm_stats['std_list']}
╠══════════════════════════════════════════════════════════╣
  ImageNet mean      (R, G, B) : {imagenet_mean}
  ImageNet std       (R, G, B) : {imagenet_std}
╠══════════════════════════════════════════════════════════╣
  Samples used    : {norm_stats['samples_used']:,}
  Pixels counted  : {norm_stats['pixels_counted']:,}
  Skipped         : {norm_stats['skipped']}
  Saved to        : {norm_path.resolve()}
╚══════════════════════════════════════════════════════════╝
""")

print("HOW TO USE IN MODEL TRAINING:")
print("─" * 50)
print("# PyTorch")
print(f"mean = {norm_stats['mean_list']}")
print(f"std  = {norm_stats['std_list']}")
print("""transforms.Normalize(mean=mean, std=std)""")
print()
print("# Keras / TensorFlow")
print(f"mean = np.array({norm_stats['mean_list']}) * 255")
print(f"std  = np.array({norm_stats['std_list']}) * 255")

In [ ]:
import json

config = {
    "class_names"   : class_names,
    "num_classes"   : len(class_names),
    "image_size"    : [224, 224],
    "splits"        : ["train", "val", "test"],
    "split_dir"     : str(SPLIT_DIR.resolve()),
    "manifest_path" : str(MANIFEST_PATH.resolve()),
    "class_weights" : class_weights,
    "norm_mean"     : norm_stats["mean_list"],
    "norm_std"      : norm_stats["std_list"]
}

config_path = Path("../config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ config.json saved → {config_path.resolve()}")
print(f"  Classes  : {config['num_classes']}")
print(f"  Mean     : {config['norm_mean']}")
print(f"  Std      : {config['norm_std']}")

In [ ]:
print("\n=== REPORTS FOLDER CHECK ===")
REPORTS_DIR = Path("../reports")

required_files = [
    "class_weights.json",
    "normalization_stats.json",
    "dataset_report.json",
    "class_distribution.png",
    "sample_image_grid.png",
    "imbalance_handling.png",
    "split_distribution.png"
]

for f in required_files:
    path = REPORTS_DIR / f
    if path.exists():
        print(f"  ✅ {f}")
    else:
        print(f"  ❌ {f} — missing")